In [1]:
#pandas for spreadsheet processing
import pandas as pd
pd.set_option('display.max_columns', None) #display all columns

#numpy for mathematical functions
import numpy as np

#geopandas to process geospatial/gis data
import geopandas as gpd

#library for connecting to the geodatabase
from sqlalchemy import create_engine, text
from geoalchemy2 import Geometry

# User input

- All human input values will need to be input in by the user into the following cells. 
- The rest of the code will run without required human intervention.

## 1. Input Link to downloaded datasets

In [2]:
# highest level of qualification by sex
# To download original dataset go to - 
# https://www.nomisweb.co.uk/query/construct/summary.asp?menuopt=200&subcomp=
edu_csv = r"N:\Geodatabase\Raw_Data\Census 2021\Highest level of qualification\30507021813828305.csv"

## 2. Input Link to LSOA 2021 shapefile
#### To download original dataset go to - 
https://geoportal.statistics.gov.uk/datasets/ons::lower-layer-super-output-areas-december-2021-boundaries-ew-bsc-v4-2/about

In [3]:
lsoa2021_shapefile = r"C:\Users\abhimanya.achara\OneDrive - priorpartners.com\Documents\UK DATA\2021 Census Data\Lower_Layer_Super_Output_Areas_(Dec_2021)_Boundaries_Full_Extent_(BFE)_EW\Lower_Layer_Super_Output_Areas_(Dec_2021)_Boundaries_Full_Extent_(BFE)_EW.shp"

## 3. Threshold to define dominant type
This defines the % threshold from the highest value of group of data columns to define which ones will be defined as dominant. 

In [4]:
threshold_value = 5

## 1. Import & Process Data

To download original dataset go to - 
https://www.nomisweb.co.uk/query/construct/summary.asp?mode=construct&version=0&dataset=2221

In [5]:
# Create Dataframes
female_edu_df = pd.read_csv(edu_csv, skiprows = 8, nrows = 35672, skip_blank_lines = True)
male_edu_df = pd.read_csv(edu_csv, skiprows = 35703, nrows = 35672, skip_blank_lines = True)

In [6]:
# Split the first column into two new columns
female_edu_df[['lsoa21cd', 'lsoa21nm']] = female_edu_df.iloc[:, 0].str.split(' : ', expand=True)
male_edu_df[['lsoa21cd', 'lsoa21nm']] = male_edu_df.iloc[:, 0].str.split(' : ', expand=True)

# Remove the column '2021 super output area - lower layer'
female_edu_df.drop(['2021 super output area - lower layer'], 1,  inplace=True)
male_edu_df.drop(['2021 super output area - lower layer'], 1, inplace=True)

# Rename columns for female_edu_df
cols_to_rename1 = female_edu_df.columns.difference(['lsoa21cd', 'lsoa21nm'])
female_edu_df.rename(columns={col: col.lower().replace(' ', '_') + '_female_count' for col in cols_to_rename1}, inplace=True)

# Rename columns for male_edu_df
cols_to_rename2 = male_edu_df.columns.difference(['lsoa21cd', 'lsoa21nm'])
male_edu_df.rename(columns={col: col.lower().replace(' ', '_') + '_male_count' for col in cols_to_rename2}, inplace=True)

# Remove the column '2021 super output area - lower layer'
female_edu_df.drop(['lsoa21nm'], 1,  inplace=True)
male_edu_df.drop(['lsoa21nm'], 1, inplace=True)

# Move 'lsoa21cd' and 'lsoa21nm' to the front of the dataframe
female_edu_df = female_edu_df[['lsoa21cd'] + [col for col in female_edu_df.columns if col not in ['lsoa21cd']]]
male_edu_df = male_edu_df[['lsoa21cd'] + [col for col in male_edu_df.columns if col not in ['lsoa21cd']]]

#Create sum totals
female_edu_df['total_female_edu_pop_count'] = female_edu_df.sum(axis=1,numeric_only=True)
male_edu_df['total_male_edu_pop_count'] = male_edu_df.sum(axis=1,numeric_only=True)

C:\Users\abhimanya.achara\AppData\Local\Temp\ipykernel_27528\2989423164.py:6: FutureWarning: In a future version of pandas all arguments of DataFrame.drop except for the argument 'labels' will be keyword-only.
  female_edu_df.drop(['2021 super output area - lower layer'], 1,  inplace=True)
C:\Users\abhimanya.achara\AppData\Local\Temp\ipykernel_27528\2989423164.py:7: FutureWarning: In a future version of pandas all arguments of DataFrame.drop except for the argument 'labels' will be keyword-only.
  male_edu_df.drop(['2021 super output area - lower layer'], 1, inplace=True)
C:\Users\abhimanya.achara\AppData\Local\Temp\ipykernel_27528\2989423164.py:18: FutureWarning: In a future version of pandas all arguments of DataFrame.drop except for the argument 'labels' will be keyword-only.
  female_edu_df.drop(['lsoa21nm'], 1,  inplace=True)
C:\Users\abhimanya.achara\AppData\Local\Temp\ipykernel_27528\2989423164.py:19: FutureWarning: In a future version of pandas all arguments of DataFrame.drop e

## 2. Feature Engineering

In [7]:
# Create percentage columns for female_edu_df
total_female_col = 'total_female_edu_pop_count'
for col in female_edu_df.columns[1:-1]:
    perc_col = col.replace('_count', '_perc')
    female_edu_df[perc_col] = (female_edu_df[col] / female_edu_df[total_female_col]) * 100

# Create percentage columns for male_edu_df
total_male_col = 'total_male_edu_pop_count'
for col in male_edu_df.columns[1:-1]:
    perc_col = col.replace('_count', '_perc')
    male_edu_df[perc_col] = (male_edu_df[col] / male_edu_df[total_male_col]) * 100

In [8]:
# Combine the two dataframes on lsoa21cd and lsoa21nm
combined_df = female_edu_df.merge(male_edu_df, on=['lsoa21cd'])

# Display the first few rows of the combined dataframe
combined_df.head()

,lsoa21cd,no_qualifications_female_count,level_1_and_entry_level_qualifications_female_count,level_2_qualifications_female_count,apprenticeship_female_count,level_3_qualifications_female_count,level_4_qualifications_or_above_female_count,other_qualifications_female_count,total_female_edu_pop_count,no_qualifications_female_perc,level_1_and_entry_level_qualifications_female_perc,level_2_qualifications_female_perc,apprenticeship_female_perc,level_3_qualifications_female_perc,level_4_qualifications_or_above_female_perc,other_qualifications_female_perc,no_qualifications_male_count,level_1_and_entry_level_qualifications_male_count,level_2_qualifications_male_count,apprenticeship_male_count,level_3_qualifications_male_count,level_4_qualifications_or_above_male_count,other_qualifications_male_count,total_male_edu_pop_count,no_qualifications_male_perc,level_1_and_entry_level_qualifications_male_perc,level_2_qualifications_male_perc,apprenticeship_male_perc,level_3_qualifications_male_perc,level_4_qualifications_or_above_male_perc,other_qualifications_male_perc
0,E01011954,248,99,181,28,183,190,22,951,26.077813,10.410095,19.032597,2.944269,19.242902,19.978970,2.313354,176,79,110,105,175,141,21,807,21.809170,9.789343,13.630731,13.011152,21.685254,17.472119,2.602230
1,E01011969,148,61,96,24,100,143,15,587,25.212947,10.391823,16.354344,4.088586,17.035775,24.361158,2.555366,97,50,50,117,119,119,14,566,17.137809,8.833922,8.833922,20.671378,21.024735,21.024735,2.473498
2,E01011970,94,46,77,11,86,150,25,489,19.222904,9.406953,15.746421,2.249489,17.586912,30.674847,5.112474,63,26,54,68,95,137,5,448,14.062500,5.803571,12.053571,15.178571,21.205357,30.580357,1.116071
3,E01011971,66,40,106,11,121,201,20,565,11.681416,7.079646,18.761062,1.946903,21.415929,35.575221,3.539823,46,31,77,63,134,154,10,515,8.932039,6.019417,14.951456,12.233010,26.019417,29.902913,1.941748
4,E01033465,94,43,124,24,167,286,14,752,12.500000,5.718085,16.489362,3.191489,22.207447,38.031915,1.861702,69,60,100,74,188,230,15,736,9.375000,8.152174,13.586957,10.054348,25.543478,31.250000,2.038043


In [9]:
# Create aggregated count columns by summing female and male counts
count_columns = [col for col in combined_df.columns if col.endswith('_female_count')]
for col in count_columns:
    base_name = col.replace('_female_count', '_count')
    male_col = col.replace('_female_count', '_male_count')
    if male_col in combined_df.columns:
        combined_df[base_name] = combined_df[col] + combined_df[male_col]

# Display the first few rows of the combined dataframe
combined_df.head()

,lsoa21cd,no_qualifications_female_count,level_1_and_entry_level_qualifications_female_count,level_2_qualifications_female_count,apprenticeship_female_count,level_3_qualifications_female_count,level_4_qualifications_or_above_female_count,other_qualifications_female_count,total_female_edu_pop_count,no_qualifications_female_perc,level_1_and_entry_level_qualifications_female_perc,level_2_qualifications_female_perc,apprenticeship_female_perc,level_3_qualifications_female_perc,level_4_qualifications_or_above_female_perc,other_qualifications_female_perc,no_qualifications_male_count,level_1_and_entry_level_qualifications_male_count,level_2_qualifications_male_count,apprenticeship_male_count,level_3_qualifications_male_count,level_4_qualifications_or_above_male_count,other_qualifications_male_count,total_male_edu_pop_count,no_qualifications_male_perc,level_1_and_entry_level_qualifications_male_perc,level_2_qualifications_male_perc,apprenticeship_male_perc,level_3_qualifications_male_perc,level_4_qualifications_or_above_male_perc,other_qualifications_male_perc,no_qualifications_count,level_1_and_entry_level_qualifications_count,level_2_qualifications_count,apprenticeship_count,level_3_qualifications_count,level_4_qualifications_or_above_count,other_qualifications_count
0,E01011954,248,99,181,28,183,190,22,951,26.077813,10.410095,19.032597,2.944269,19.242902,19.978970,2.313354,176,79,110,105,175,141,21,807,21.809170,9.789343,13.630731,13.011152,21.685254,17.472119,2.602230,424,178,291,133,358,331,43
1,E01011969,148,61,96,24,100,143,15,587,25.212947,10.391823,16.354344,4.088586,17.035775,24.361158,2.555366,97,50,50,117,119,119,14,566,17.137809,8.833922,8.833922,20.671378,21.024735,21.024735,2.473498,245,111,146,141,219,262,29
2,E01011970,94,46,77,11,86,150,25,489,19.222904,9.406953,15.746421,2.249489,17.586912,30.674847,5.112474,63,26,54,68,95,137,5,448,14.062500,5.803571,12.053571,15.178571,21.205357,30.580357,1.116071,157,72,131,79,181,287,30
3,E01011971,66,40,106,11,121,201,20,565,11.681416,7.079646,18.761062,1.946903,21.415929,35.575221,3.539823,46,31,77,63,134,154,10,515,8.932039,6.019417,14.951456,12.233010,26.019417,29.902913,1.941748,112,71,183,74,255,355,30
4,E01033465,94,43,124,24,167,286,14,752,12.500000,5.718085,16.489362,3.191489,22.207447,38.031915,1.861702,69,60,100,74,188,230,15,736,9.375000,8.152174,13.586957,10.054348,25.543478,31.250000,2.038043,163,103,224,98,355,516,29


In [10]:
# Create percentage columns for female_pop_df
combined_df['total_edu_pop_count'] = combined_df['total_female_edu_pop_count'] + combined_df['total_male_edu_pop_count']
total_col = 'total_edu_pop_count'
for col in combined_df.columns[-8:]:
    perc_col = col.replace('_count', '_perc')
    combined_df[perc_col] = (combined_df[col] / combined_df[total_col]) * 100

combined_df.drop(['total_edu_pop_perc'],1,inplace = True)
combined_df.head()

C:\Users\abhimanya.achara\AppData\Local\Temp\ipykernel_27528\2567787110.py:8: FutureWarning: In a future version of pandas all arguments of DataFrame.drop except for the argument 'labels' will be keyword-only.
  combined_df.drop(['total_edu_pop_perc'],1,inplace = True)


,lsoa21cd,no_qualifications_female_count,level_1_and_entry_level_qualifications_female_count,level_2_qualifications_female_count,apprenticeship_female_count,level_3_qualifications_female_count,level_4_qualifications_or_above_female_count,other_qualifications_female_count,total_female_edu_pop_count,no_qualifications_female_perc,level_1_and_entry_level_qualifications_female_perc,level_2_qualifications_female_perc,apprenticeship_female_perc,level_3_qualifications_female_perc,level_4_qualifications_or_above_female_perc,other_qualifications_female_perc,no_qualifications_male_count,level_1_and_entry_level_qualifications_male_count,level_2_qualifications_male_count,apprenticeship_male_count,level_3_qualifications_male_count,level_4_qualifications_or_above_male_count,other_qualifications_male_count,total_male_edu_pop_count,no_qualifications_male_perc,level_1_and_entry_level_qualifications_male_perc,level_2_qualifications_male_perc,apprenticeship_male_perc,level_3_qualifications_male_perc,level_4_qualifications_or_above_male_perc,other_qualifications_male_perc,no_qualifications_count,level_1_and_entry_level_qualifications_count,level_2_qualifications_count,apprenticeship_count,level_3_qualifications_count,level_4_qualifications_or_above_count,other_qualifications_count,total_edu_pop_count,no_qualifications_perc,level_1_and_entry_level_qualifications_perc,level_2_qualifications_perc,apprenticeship_perc,level_3_qualifications_perc,level_4_qualifications_or_above_perc,other_qualifications_perc
0,E01011954,248,99,181,28,183,190,22,951,26.077813,10.410095,19.032597,2.944269,19.242902,19.978970,2.313354,176,79,110,105,175,141,21,807,21.809170,9.789343,13.630731,13.011152,21.685254,17.472119,2.602230,424,178,291,133,358,331,43,1758,24.118316,10.125142,16.552901,7.565415,20.364050,18.828214,2.445961
1,E01011969,148,61,96,24,100,143,15,587,25.212947,10.391823,16.354344,4.088586,17.035775,24.361158,2.555366,97,50,50,117,119,119,14,566,17.137809,8.833922,8.833922,20.671378,21.024735,21.024735,2.473498,245,111,146,141,219,262,29,1153,21.248916,9.627060,12.662619,12.228968,18.993929,22.723330,2.515178
2,E01011970,94,46,77,11,86,150,25,489,19.222904,9.406953,15.746421,2.249489,17.586912,30.674847,5.112474,63,26,54,68,95,137,5,448,14.062500,5.803571,12.053571,15.178571,21.205357,30.580357,1.116071,157,72,131,79,181,287,30,937,16.755603,7.684098,13.980790,8.431163,19.316969,30.629669,3.201708
3,E01011971,66,40,106,11,121,201,20,565,11.681416,7.079646,18.761062,1.946903,21.415929,35.575221,3.539823,46,31,77,63,134,154,10,515,8.932039,6.019417,14.951456,12.233010,26.019417,29.902913,1.941748,112,71,183,74,255,355,30,1080,10.370370,6.574074,16.944444,6.851852,23.611111,32.870370,2.777778
4,E01033465,94,43,124,24,167,286,14,752,12.500000,5.718085,16.489362,3.191489,22.207447,38.031915,1.861702,69,60,100,74,188,230,15,736,9.375000,8.152174,13.586957,10.054348,25.543478,31.250000,2.038043,163,103,224,98,355,516,29,1488,10.954301,6.922043,15.053763,6.586022,23.857527,34.677419,1.948925


## 4. Load GIS shapefile 

In [11]:
# Attempt to read from the server first, fallback to local path if it fails
lsoa2021boundary_df = gpd.read_file(lsoa2021_shapefile)
print("Shapefile loaded successfully from the server.")

# Display the first few rows
lsoa2021boundary_df.head()

Shapefile loaded successfully from the server.


,FID,LSOA21CD,LSOA21NM,Shape__Are,Shape__Len,GlobalID,geometry
0,1,E01000001,City of London 001A,129865.314476,2635.767993,4dd060c0-a44a-4c62-aca3-b0c88c0bf0d0,"POLYGON ((532151.537 181867.433, 532152.500 18..."
1,2,E01000002,City of London 001B,228419.782242,2707.816821,d0a47b5d-5649-4242-a8e7-462078975c1d,"POLYGON ((532634.497 181926.016, 532632.048 18..."
2,3,E01000003,City of London 001C,59054.204697,1224.573160,2904cc10-e994-4a6e-b11c-06be7fbd74d2,"POLYGON ((532153.703 182165.155, 532158.250 18..."
3,4,E01000005,City of London 001E,189577.709503,2275.805344,bfcf5958-f052-4388-8efd-75bb808d8b83,"POLYGON ((533619.062 181402.364, 533639.868 18..."
4,5,E01000006,Barking and Dagenham 016A,146536.995750,1966.092607,64a24862-79c1-4c8c-ab7b-d9cec1d3c0c6,"POLYGON ((545126.852 184310.838, 545145.213 18..."


## 5. GIS data management

### Remove Rename fields

In [12]:
#Drop and rename fields
lsoa2021boundary_df.drop(['Shape__Are', 'Shape__Len', 'GlobalID'], 1, inplace = True)
lsoa2021boundary_df.rename(columns = {'LSOA21CD':'lsoa21cd','LSOA21NM':'lsoa21nm'}, inplace = True)
lsoa2021boundary_df.head()

C:\Users\abhimanya.achara\AppData\Local\Temp\ipykernel_27528\2026090638.py:2: FutureWarning: In a future version of pandas all arguments of DataFrame.drop except for the argument 'labels' will be keyword-only.
  lsoa2021boundary_df.drop(['Shape__Are', 'Shape__Len', 'GlobalID'], 1, inplace = True)


,FID,lsoa21cd,lsoa21nm,geometry
0,1,E01000001,City of London 001A,"POLYGON ((532151.537 181867.433, 532152.500 18..."
1,2,E01000002,City of London 001B,"POLYGON ((532634.497 181926.016, 532632.048 18..."
2,3,E01000003,City of London 001C,"POLYGON ((532153.703 182165.155, 532158.250 18..."
3,4,E01000005,City of London 001E,"POLYGON ((533619.062 181402.364, 533639.868 18..."
4,5,E01000006,Barking and Dagenham 016A,"POLYGON ((545126.852 184310.838, 545145.213 18..."


### Combine with boundary lookup table

#### LSOA21 to MSOA21 to LAD22

In [13]:
# Define the file path for msoa11 to msoa21 lookup
file_path = r"N:\Geodatabase\Raw_Data\Look up tables\OA21_LSOA21_MSOA21_LAD22_EW_LU.csv"

# Read the Excel file
lsoalookup_df = pd.read_csv(file_path)  # Reads the first sheet by default

#drop oa column
lsoalookup_df.drop(['oa21cd','lsoa21nm'],1,inplace = True)

#remove duplicates
lsoalookup_df = lsoalookup_df.drop_duplicates(subset='lsoa21cd', keep='first')

# Display the first few rows
lsoalookup_df.head()

C:\Users\abhimanya.achara\AppData\Local\Temp\ipykernel_27528\1268765203.py:8: FutureWarning: In a future version of pandas all arguments of DataFrame.drop except for the argument 'labels' will be keyword-only.
  lsoalookup_df.drop(['oa21cd','lsoa21nm'],1,inplace = True)


,lsoa21cd,msoa21cd,msoa21nm,lad22cd,lad22nm
0,E01000001,E02000001,City of London 001,E09000001,City of London
4,E01000003,E02000001,City of London 001,E09000001,City of London
6,E01000002,E02000001,City of London 001,E09000001,City of London
12,E01032739,E02000001,City of London 001,E09000001,City of London
13,E01032740,E02000001,City of London 001,E09000001,City of London


In [14]:
lsoa2021boundary_df = pd.merge(lsoa2021boundary_df, lsoalookup_df, how = 'left', on = 'lsoa21cd')
lsoa2021boundary_df.head()

,FID,lsoa21cd,lsoa21nm,geometry,msoa21cd,msoa21nm,lad22cd,lad22nm
0,1,E01000001,City of London 001A,"POLYGON ((532151.537 181867.433, 532152.500 18...",E02000001,City of London 001,E09000001,City of London
1,2,E01000002,City of London 001B,"POLYGON ((532634.497 181926.016, 532632.048 18...",E02000001,City of London 001,E09000001,City of London
2,3,E01000003,City of London 001C,"POLYGON ((532153.703 182165.155, 532158.250 18...",E02000001,City of London 001,E09000001,City of London
3,4,E01000005,City of London 001E,"POLYGON ((533619.062 181402.364, 533639.868 18...",E02000001,City of London 001,E09000001,City of London
4,5,E01000006,Barking and Dagenham 016A,"POLYGON ((545126.852 184310.838, 545145.213 18...",E02000017,Barking and Dagenham 016,E09000002,Barking and Dagenham


#### LSOA21 to WARD22

In [15]:
# Define the file path for msoa21 to ward lookup
file_path = r"N:\Geodatabase\Raw_Data\Look up tables\Lower_Layer_Super_Output_Area_(2021)_to_Ward_(2022)_to_LAD_(2022)_Lookup_in_England_and_Wales_v3.csv"

# Read the Excel file
lsoawardlookup_df = pd.read_csv(file_path)  # Reads the first sheet by default

#drop unwanted fields
lsoawardlookup_df.drop(['LSOA21NM','LSOA21NMW','WD22NMW','LAD22NMW','ObjectId','LAD22CD','LAD22NM'],1,inplace = True)

# Rename fields
lsoawardlookup_df.rename(
    columns={
        'LSOA21CD': 'lsoa21cd',
        'WD22CD': 'wd22cd',
        'WD22NM': 'wd22nm',
        }, 
    inplace=True
)

# Display the first few rows
lsoawardlookup_df.head()

C:\Users\abhimanya.achara\AppData\Local\Temp\ipykernel_27528\1707502962.py:8: FutureWarning: In a future version of pandas all arguments of DataFrame.drop except for the argument 'labels' will be keyword-only.
  lsoawardlookup_df.drop(['LSOA21NM','LSOA21NMW','WD22NMW','LAD22NMW','ObjectId','LAD22CD','LAD22NM'],1,inplace = True)


,lsoa21cd,wd22cd,wd22nm
0,E01004766,E05000650,Astley Bridge
1,E01005347,E05000722,Chadderton South
2,E01004890,E05000661,Horwich North East
3,E01004770,E05000650,Astley Bridge
4,E01004888,E05000661,Horwich North East


In [16]:
lsoa2021boundary_df = pd.merge(lsoa2021boundary_df, lsoawardlookup_df, how = 'inner', on = 'lsoa21cd')
lsoa2021boundary_df.head()

,FID,lsoa21cd,lsoa21nm,geometry,msoa21cd,msoa21nm,lad22cd,lad22nm,wd22cd,wd22nm
0,1,E01000001,City of London 001A,"POLYGON ((532151.537 181867.433, 532152.500 18...",E02000001,City of London 001,E09000001,City of London,E05009288,Aldersgate
1,2,E01000002,City of London 001B,"POLYGON ((532634.497 181926.016, 532632.048 18...",E02000001,City of London 001,E09000001,City of London,E05009302,Cripplegate
2,3,E01000003,City of London 001C,"POLYGON ((532153.703 182165.155, 532158.250 18...",E02000001,City of London 001,E09000001,City of London,E05009302,Cripplegate
3,4,E01000005,City of London 001E,"POLYGON ((533619.062 181402.364, 533639.868 18...",E02000001,City of London 001,E09000001,City of London,E05009308,Portsoken
4,5,E01000006,Barking and Dagenham 016A,"POLYGON ((545126.852 184310.838, 545145.213 18...",E02000017,Barking and Dagenham 016,E09000002,Barking and Dagenham,E05014066,Northbury


#### LAD22 to REGION22

In [17]:
# Define the file path for lad to region
file_path = r"N:\Geodatabase\Raw_Data\Look up tables\Local_Authority_District_to_Region_(December_2022)_Lookup_in_England.csv"

# Read the Excel file
ladregionlookup_df = pd.read_csv(file_path)  # Reads the first sheet by default

#drop unwanted fields
ladregionlookup_df.drop(['LAD22NM','ObjectId'],1,inplace = True)

# Rename fields
ladregionlookup_df.rename(
    columns={
        'LAD22CD': 'lad22cd',
        'RGN22CD': 'rgn22cd',
        'RGN22NM': 'rgn22nm'
       }, 
    inplace=True
)

# Display the first few rows
ladregionlookup_df.head()

C:\Users\abhimanya.achara\AppData\Local\Temp\ipykernel_27528\4188529333.py:8: FutureWarning: In a future version of pandas all arguments of DataFrame.drop except for the argument 'labels' will be keyword-only.
  ladregionlookup_df.drop(['LAD22NM','ObjectId'],1,inplace = True)


,lad22cd,rgn22cd,rgn22nm
0,E06000001,E12000001,North East
1,E06000002,E12000001,North East
2,E06000003,E12000001,North East
3,E06000004,E12000001,North East
4,E06000005,E12000001,North East


In [18]:
lsoa2021boundary_df = pd.merge(lsoa2021boundary_df, ladregionlookup_df, how = 'left', on = 'lad22cd')
lsoa2021boundary_df.head()

,FID,lsoa21cd,lsoa21nm,geometry,msoa21cd,msoa21nm,lad22cd,lad22nm,wd22cd,wd22nm,rgn22cd,rgn22nm
0,1,E01000001,City of London 001A,"POLYGON ((532151.537 181867.433, 532152.500 18...",E02000001,City of London 001,E09000001,City of London,E05009288,Aldersgate,E12000007,London
1,2,E01000002,City of London 001B,"POLYGON ((532634.497 181926.016, 532632.048 18...",E02000001,City of London 001,E09000001,City of London,E05009302,Cripplegate,E12000007,London
2,3,E01000003,City of London 001C,"POLYGON ((532153.703 182165.155, 532158.250 18...",E02000001,City of London 001,E09000001,City of London,E05009302,Cripplegate,E12000007,London
3,4,E01000005,City of London 001E,"POLYGON ((533619.062 181402.364, 533639.868 18...",E02000001,City of London 001,E09000001,City of London,E05009308,Portsoken,E12000007,London
4,5,E01000006,Barking and Dagenham 016A,"POLYGON ((545126.852 184310.838, 545145.213 18...",E02000017,Barking and Dagenham 016,E09000002,Barking and Dagenham,E05014066,Northbury,E12000007,London


### Add data management fields

In [19]:
# Add new data management fields with specified values
lsoa2021boundary_df['data_source'] = "Census 2021 (ONS Nomis - Official Census and Labour Market Statistics)"
lsoa2021boundary_df['data_resolution'] = "LSOA 2021"
lsoa2021boundary_df['data_time_period'] = pd.to_datetime("2021-02-21")  # Convert to date format
lsoa2021boundary_df['data_web_link'] = "https://www.nomisweb.co.uk/"
lsoa2021boundary_df.head()

,FID,lsoa21cd,lsoa21nm,geometry,msoa21cd,msoa21nm,lad22cd,lad22nm,wd22cd,wd22nm,rgn22cd,rgn22nm,data_source,data_resolution,data_time_period,data_web_link
0,1,E01000001,City of London 001A,"POLYGON ((532151.537 181867.433, 532152.500 18...",E02000001,City of London 001,E09000001,City of London,E05009288,Aldersgate,E12000007,London,Census 2021 (ONS Nomis - Official Census and L...,LSOA 2021,2021-02-21,https://www.nomisweb.co.uk/
1,2,E01000002,City of London 001B,"POLYGON ((532634.497 181926.016, 532632.048 18...",E02000001,City of London 001,E09000001,City of London,E05009302,Cripplegate,E12000007,London,Census 2021 (ONS Nomis - Official Census and L...,LSOA 2021,2021-02-21,https://www.nomisweb.co.uk/
2,3,E01000003,City of London 001C,"POLYGON ((532153.703 182165.155, 532158.250 18...",E02000001,City of London 001,E09000001,City of London,E05009302,Cripplegate,E12000007,London,Census 2021 (ONS Nomis - Official Census and L...,LSOA 2021,2021-02-21,https://www.nomisweb.co.uk/
3,4,E01000005,City of London 001E,"POLYGON ((533619.062 181402.364, 533639.868 18...",E02000001,City of London 001,E09000001,City of London,E05009308,Portsoken,E12000007,London,Census 2021 (ONS Nomis - Official Census and L...,LSOA 2021,2021-02-21,https://www.nomisweb.co.uk/
4,5,E01000006,Barking and Dagenham 016A,"POLYGON ((545126.852 184310.838, 545145.213 18...",E02000017,Barking and Dagenham 016,E09000002,Barking and Dagenham,E05014066,Northbury,E12000007,London,Census 2021 (ONS Nomis - Official Census and L...,LSOA 2021,2021-02-21,https://www.nomisweb.co.uk/


### Update Area

In [20]:
#Update Area in Hectares
# Ensure the dataframe has a valid geometry column
if 'geometry' in lsoa2021boundary_df.columns:
    # Calculate the area in square meters and convert to hectares (1 hectare = 10,000 m²)
    lsoa2021boundary_df['area_ha'] = lsoa2021boundary_df.geometry.area / 10_000

    # Print confirmation message
    print("Field 'area_ha' has been added and updated successfully.")
else:
    print("Error: No 'geometry' column found in lsoa2021boundary_df. Ensure it's a GeoDataFrame with valid geometries.")
    
lsoa2021boundary_df.head()

Field 'area_ha' has been added and updated successfully.


,FID,lsoa21cd,lsoa21nm,geometry,msoa21cd,msoa21nm,lad22cd,lad22nm,wd22cd,wd22nm,rgn22cd,rgn22nm,data_source,data_resolution,data_time_period,data_web_link,area_ha
0,1,E01000001,City of London 001A,"POLYGON ((532151.537 181867.433, 532152.500 18...",E02000001,City of London 001,E09000001,City of London,E05009288,Aldersgate,E12000007,London,Census 2021 (ONS Nomis - Official Census and L...,LSOA 2021,2021-02-21,https://www.nomisweb.co.uk/,12.986531
1,2,E01000002,City of London 001B,"POLYGON ((532634.497 181926.016, 532632.048 18...",E02000001,City of London 001,E09000001,City of London,E05009302,Cripplegate,E12000007,London,Census 2021 (ONS Nomis - Official Census and L...,LSOA 2021,2021-02-21,https://www.nomisweb.co.uk/,22.841978
2,3,E01000003,City of London 001C,"POLYGON ((532153.703 182165.155, 532158.250 18...",E02000001,City of London 001,E09000001,City of London,E05009302,Cripplegate,E12000007,London,Census 2021 (ONS Nomis - Official Census and L...,LSOA 2021,2021-02-21,https://www.nomisweb.co.uk/,5.905420
3,4,E01000005,City of London 001E,"POLYGON ((533619.062 181402.364, 533639.868 18...",E02000001,City of London 001,E09000001,City of London,E05009308,Portsoken,E12000007,London,Census 2021 (ONS Nomis - Official Census and L...,LSOA 2021,2021-02-21,https://www.nomisweb.co.uk/,18.957771
4,5,E01000006,Barking and Dagenham 016A,"POLYGON ((545126.852 184310.838, 545145.213 18...",E02000017,Barking and Dagenham 016,E09000002,Barking and Dagenham,E05014066,Northbury,E12000007,London,Census 2021 (ONS Nomis - Official Census and L...,LSOA 2021,2021-02-21,https://www.nomisweb.co.uk/,14.653700


# 7. Combine Geometry and data

In [21]:
census2021_education_lsoa2021_gdb_df = pd.merge(lsoa2021boundary_df,combined_df, how = 'left', on='lsoa21cd')
census2021_education_lsoa2021_gdb_df.head()

,FID,lsoa21cd,lsoa21nm,geometry,msoa21cd,msoa21nm,lad22cd,lad22nm,wd22cd,wd22nm,rgn22cd,rgn22nm,data_source,data_resolution,data_time_period,data_web_link,area_ha,no_qualifications_female_count,level_1_and_entry_level_qualifications_female_count,level_2_qualifications_female_count,apprenticeship_female_count,level_3_qualifications_female_count,level_4_qualifications_or_above_female_count,other_qualifications_female_count,total_female_edu_pop_count,no_qualifications_female_perc,level_1_and_entry_level_qualifications_female_perc,level_2_qualifications_female_perc,apprenticeship_female_perc,level_3_qualifications_female_perc,level_4_qualifications_or_above_female_perc,other_qualifications_female_perc,no_qualifications_male_count,level_1_and_entry_level_qualifications_male_count,level_2_qualifications_male_count,apprenticeship_male_count,level_3_qualifications_male_count,level_4_qualifications_or_above_male_count,other_qualifications_male_count,total_male_edu_pop_count,no_qualifications_male_perc,level_1_and_entry_level_qualifications_male_perc,level_2_qualifications_male_perc,apprenticeship_male_perc,level_3_qualifications_male_perc,level_4_qualifications_or_above_male_perc,other_qualifications_male_perc,no_qualifications_count,level_1_and_entry_level_qualifications_count,level_2_qualifications_count,apprenticeship_count,level_3_qualifications_count,level_4_qualifications_or_above_count,other_qualifications_count,total_edu_pop_count,no_qualifications_perc,level_1_and_entry_level_qualifications_perc,level_2_qualifications_perc,apprenticeship_perc,level_3_qualifications_perc,level_4_qualifications_or_above_perc,other_qualifications_perc
0,1,E01000001,City of London 001A,"POLYGON ((532151.537 181867.433, 532152.500 18...",E02000001,City of London 001,E09000001,City of London,E05009288,Aldersgate,E12000007,London,Census 2021 (ONS Nomis - Official Census and L...,LSOA 2021,2021-02-21,https://www.nomisweb.co.uk/,12.986531,15,11,31,5,34,525,7,628,2.388535,1.751592,4.936306,0.796178,5.414013,83.598726,1.114650,17,16,22,8,52,602,9,726,2.341598,2.203857,3.030303,1.101928,7.162534,82.920110,1.239669,32,27,53,13,86,1127,16,1354,2.363368,1.994092,3.914328,0.960118,6.351551,83.234860,1.181684
1,2,E01000002,City of London 001B,"POLYGON ((532634.497 181926.016, 532632.048 18...",E02000001,City of London 001,E09000001,City of London,E05009302,Cripplegate,E12000007,London,Census 2021 (ONS Nomis - Official Census and L...,LSOA 2021,2021-02-21,https://www.nomisweb.co.uk/,22.841978,15,12,27,1,40,468,6,569,2.636204,2.108963,4.745167,0.175747,7.029877,82.249561,1.054482,6,16,25,3,47,618,13,728,0.824176,2.197802,3.434066,0.412088,6.456044,84.890110,1.785714,21,28,52,4,87,1086,19,1297,1.619121,2.158828,4.009252,0.308404,6.707787,83.731689,1.464919
2,3,E01000003,City of London 001C,"POLYGON ((532153.703 182165.155, 532158.250 18...",E02000001,City of London 001,E09000001,City of London,E05009302,Cripplegate,E12000007,London,Census 2021 (ONS Nomis - Official Census and L...,LSOA 2021,2021-02-21,https://www.nomisweb.co.uk/,5.905420,63,31,45,4,47,490,15,695,9.064748,4.460432,6.474820,0.575540,6.762590,70.503597,2.158273,69,30,45,8,73,565,20,810,8.518519,3.703704,5.555556,0.987654,9.012346,69.753086,2.469136,132,61,90,12,120,1055,35,1505,8.770764,4.053156,5.980066,0.797342,7.973422,70.099668,2.325581
3,4,E01000005,City of London 001E,"POLYGON ((533619.062 181402.364, 533639.868 18...",E02000001,City of London 001,E09000001,City of London,E05009308,Portsoken,E12000007,London,Census 2021 (ONS Nomis - Official Census and L...,LSOA 2021,2021-02-21,https://www.nomisweb.co.uk/,18.957771,105,31,50,12,70,188,13,469,22.388060,6.609808,10.660981,2.558635,14.925373,40.085288,2.771855,93,33,70,20,60,199,17,492,18.902439,6.707317,14.227642,4.065041,12.195122,40.447154,3.455285,198,64,120,32,130,387,30,961,20.603538,6.659729,12.486993,3.329865,13.527575,40.270552,3.121748
4,5,E01000006,Barking and Dagenham 016A,"POLYGON ((545126.852 184310.838, 545145.213 18...",E02000017,

# 8. Final tweaks

In [22]:
# Reorder columns in the combined dataframe

final_column_order = ['FID',
 'lsoa21cd',
 'lsoa21nm',
 'geometry',
 'msoa21cd',
 'msoa21nm',
 'lad22cd',
 'lad22nm',
 'wd22cd',
 'wd22nm',
 'rgn22cd',
 'rgn22nm',
 'data_source',
 'data_resolution',
 'data_time_period',
 'data_web_link',
 'area_ha',
 'no_qualifications_count',
 'level_1_and_entry_level_qualifications_count',
 'level_2_qualifications_count',
 'apprenticeship_count',
 'level_3_qualifications_count',
 'level_4_qualifications_or_above_count',
 'other_qualifications_count',
 'total_edu_pop_count',
 'no_qualifications_perc',
 'level_1_and_entry_level_qualifications_perc',
 'level_2_qualifications_perc',
 'apprenticeship_perc',
 'level_3_qualifications_perc',
 'level_4_qualifications_or_above_perc',
 'other_qualifications_perc',
 'no_qualifications_female_count',
 'level_1_and_entry_level_qualifications_female_count',
 'level_2_qualifications_female_count',
 'apprenticeship_female_count',
 'level_3_qualifications_female_count',
 'level_4_qualifications_or_above_female_count',
 'other_qualifications_female_count',
 'total_female_edu_pop_count',
 'no_qualifications_female_perc',
 'level_1_and_entry_level_qualifications_female_perc',
 'level_2_qualifications_female_perc',
 'apprenticeship_female_perc',
 'level_3_qualifications_female_perc',
 'level_4_qualifications_or_above_female_perc',
 'other_qualifications_female_perc',
 'no_qualifications_male_count',
 'level_1_and_entry_level_qualifications_male_count',
 'level_2_qualifications_male_count',
 'apprenticeship_male_count',
 'level_3_qualifications_male_count',
 'level_4_qualifications_or_above_male_count',
 'other_qualifications_male_count',
 'total_male_edu_pop_count',
 'no_qualifications_male_perc',
 'level_1_and_entry_level_qualifications_male_perc',
 'level_2_qualifications_male_perc',
 'apprenticeship_male_perc',
 'level_3_qualifications_male_perc',
 'level_4_qualifications_or_above_male_perc',
 'other_qualifications_male_perc',
 ]

census2021_education_lsoa2021_gdb_df = census2021_education_lsoa2021_gdb_df[final_column_order]

census2021_education_lsoa2021_gdb_df.head()

,FID,lsoa21cd,lsoa21nm,geometry,msoa21cd,msoa21nm,lad22cd,lad22nm,wd22cd,wd22nm,rgn22cd,rgn22nm,data_source,data_resolution,data_time_period,data_web_link,area_ha,no_qualifications_count,level_1_and_entry_level_qualifications_count,level_2_qualifications_count,apprenticeship_count,level_3_qualifications_count,level_4_qualifications_or_above_count,other_qualifications_count,total_edu_pop_count,no_qualifications_perc,level_1_and_entry_level_qualifications_perc,level_2_qualifications_perc,apprenticeship_perc,level_3_qualifications_perc,level_4_qualifications_or_above_perc,other_qualifications_perc,no_qualifications_female_count,level_1_and_entry_level_qualifications_female_count,level_2_qualifications_female_count,apprenticeship_female_count,level_3_qualifications_female_count,level_4_qualifications_or_above_female_count,other_qualifications_female_count,total_female_edu_pop_count,no_qualifications_female_perc,level_1_and_entry_level_qualifications_female_perc,level_2_qualifications_female_perc,apprenticeship_female_perc,level_3_qualifications_female_perc,level_4_qualifications_or_above_female_perc,other_qualifications_female_perc,no_qualifications_male_count,level_1_and_entry_level_qualifications_male_count,level_2_qualifications_male_count,apprenticeship_male_count,level_3_qualifications_male_count,level_4_qualifications_or_above_male_count,other_qualifications_male_count,total_male_edu_pop_count,no_qualifications_male_perc,level_1_and_entry_level_qualifications_male_perc,level_2_qualifications_male_perc,apprenticeship_male_perc,level_3_qualifications_male_perc,level_4_qualifications_or_above_male_perc,other_qualifications_male_perc
0,1,E01000001,City of London 001A,"POLYGON ((532151.537 181867.433, 532152.500 18...",E02000001,City of London 001,E09000001,City of London,E05009288,Aldersgate,E12000007,London,Census 2021 (ONS Nomis - Official Census and L...,LSOA 2021,2021-02-21,https://www.nomisweb.co.uk/,12.986531,32,27,53,13,86,1127,16,1354,2.363368,1.994092,3.914328,0.960118,6.351551,83.234860,1.181684,15,11,31,5,34,525,7,628,2.388535,1.751592,4.936306,0.796178,5.414013,83.598726,1.114650,17,16,22,8,52,602,9,726,2.341598,2.203857,3.030303,1.101928,7.162534,82.920110,1.239669
1,2,E01000002,City of London 001B,"POLYGON ((532634.497 181926.016, 532632.048 18...",E02000001,City of London 001,E09000001,City of London,E05009302,Cripplegate,E12000007,London,Census 2021 (ONS Nomis - Official Census and L...,LSOA 2021,2021-02-21,https://www.nomisweb.co.uk/,22.841978,21,28,52,4,87,1086,19,1297,1.619121,2.158828,4.009252,0.308404,6.707787,83.731689,1.464919,15,12,27,1,40,468,6,569,2.636204,2.108963,4.745167,0.175747,7.029877,82.249561,1.054482,6,16,25,3,47,618,13,728,0.824176,2.197802,3.434066,0.412088,6.456044,84.890110,1.785714
2,3,E01000003,City of London 001C,"POLYGON ((532153.703 182165.155, 532158.250 18...",E02000001,City of London 001,E09000001,City of London,E05009302,Cripplegate,E12000007,London,Census 2021 (ONS Nomis - Official Census and L...,LSOA 2021,2021-02-21,https://www.nomisweb.co.uk/,5.905420,132,61,90,12,120,1055,35,1505,8.770764,4.053156,5.980066,0.797342,7.973422,70.099668,2.325581,63,31,45,4,47,490,15,695,9.064748,4.460432,6.474820,0.575540,6.762590,70.503597,2.158273,69,30,45,8,73,565,20,810,8.518519,3.703704,5.555556,0.987654,9.012346,69.753086,2.469136
3,4,E01000005,City of London 001E,"POLYGON ((533619.062 181402.364, 533639.868 18...",E02000001,City of London 001,E09000001,City of London,E05009308,Portsoken,E12000007,London,Census 2021 (ONS Nomis - Official Census and L...,LSOA 2021,2021-02-21,https://www.nomisweb.co.uk/,18.957771,198,64,120,32,130,387,30,961,20.603538,6.659729,12.486993,3.329865,13.527575,40.270552,3.121748,105,31,50,12,70,188,13,469,22.388060,6.609808,10.660981,2.558635,14.925373,40.085288,2.771855,93,33,70,20,60,199,17,492,18.902439,6.707317,14.227642,4.065041,12.195122,40.447154,3.455285
4,5,E01000006,Barking and Dagenham 016A,"POLYGON ((545126.852 184310.838, 545145.213 18...",E02000017,

# 8. Create dominant field

In [23]:
def determine_dominant_group(row):
    columns = [
 'no_qualifications_perc',
 'level_1_and_entry_level_qualifications_perc',
 'level_2_qualifications_perc',
 'apprenticeship_perc',
 'level_3_qualifications_perc',
 'level_4_qualifications_or_above_perc',
 'other_qualifications_perc',
    ]
    
    max_value = max(row[col] for col in columns)
    threshold = max_value - threshold_value
    
    dominant_groups = [col.replace('_perc', '') for col in columns if row[col] >= threshold]
    
    return ', '.join(dominant_groups)

census2021_education_lsoa2021_gdb_df['dominant_education_group'] = census2021_education_lsoa2021_gdb_df.apply(determine_dominant_group, axis=1)

# Display the first few rows of the updated dataframe
census2021_education_lsoa2021_gdb_df.head()

,FID,lsoa21cd,lsoa21nm,geometry,msoa21cd,msoa21nm,lad22cd,lad22nm,wd22cd,wd22nm,rgn22cd,rgn22nm,data_source,data_resolution,data_time_period,data_web_link,area_ha,no_qualifications_count,level_1_and_entry_level_qualifications_count,level_2_qualifications_count,apprenticeship_count,level_3_qualifications_count,level_4_qualifications_or_above_count,other_qualifications_count,total_edu_pop_count,no_qualifications_perc,level_1_and_entry_level_qualifications_perc,level_2_qualifications_perc,apprenticeship_perc,level_3_qualifications_perc,level_4_qualifications_or_above_perc,other_qualifications_perc,no_qualifications_female_count,level_1_and_entry_level_qualifications_female_count,level_2_qualifications_female_count,apprenticeship_female_count,level_3_qualifications_female_count,level_4_qualifications_or_above_female_count,other_qualifications_female_count,total_female_edu_pop_count,no_qualifications_female_perc,level_1_and_entry_level_qualifications_female_perc,level_2_qualifications_female_perc,apprenticeship_female_perc,level_3_qualifications_female_perc,level_4_qualifications_or_above_female_perc,other_qualifications_female_perc,no_qualifications_male_count,level_1_and_entry_level_qualifications_male_count,level_2_qualifications_male_count,apprenticeship_male_count,level_3_qualifications_male_count,level_4_qualifications_or_above_male_count,other_qualifications_male_count,total_male_edu_pop_count,no_qualifications_male_perc,level_1_and_entry_level_qualifications_male_perc,level_2_qualifications_male_perc,apprenticeship_male_perc,level_3_qualifications_male_perc,level_4_qualifications_or_above_male_perc,other_qualifications_male_perc,dominant_education_group
0,1,E01000001,City of London 001A,"POLYGON ((532151.537 181867.433, 532152.500 18...",E02000001,City of London 001,E09000001,City of London,E05009288,Aldersgate,E12000007,London,Census 2021 (ONS Nomis - Official Census and L...,LSOA 2021,2021-02-21,https://www.nomisweb.co.uk/,12.986531,32,27,53,13,86,1127,16,1354,2.363368,1.994092,3.914328,0.960118,6.351551,83.234860,1.181684,15,11,31,5,34,525,7,628,2.388535,1.751592,4.936306,0.796178,5.414013,83.598726,1.114650,17,16,22,8,52,602,9,726,2.341598,2.203857,3.030303,1.101928,7.162534,82.920110,1.239669,level_4_qualifications_or_above
1,2,E01000002,City of London 001B,"POLYGON ((532634.497 181926.016, 532632.048 18...",E02000001,City of London 001,E09000001,City of London,E05009302,Cripplegate,E12000007,London,Census 2021 (ONS Nomis - Official Census and L...,LSOA 2021,2021-02-21,https://www.nomisweb.co.uk/,22.841978,21,28,52,4,87,1086,19,1297,1.619121,2.158828,4.009252,0.308404,6.707787,83.731689,1.464919,15,12,27,1,40,468,6,569,2.636204,2.108963,4.745167,0.175747,7.029877,82.249561,1.054482,6,16,25,3,47,618,13,728,0.824176,2.197802,3.434066,0.412088,6.456044,84.890110,1.785714,level_4_qualifications_or_above
2,3,E01000003,City of London 001C,"POLYGON ((532153.703 182165.155, 532158.250 18...",E02000001,City of London 001,E09000001,City of London,E05009302,Cripplegate,E12000007,London,Census 2021 (ONS Nomis - Official Census and L...,LSOA 2021,2021-02-21,https://www.nomisweb.co.uk/,5.905420,132,61,90,12,120,1055,35,1505,8.770764,4.053156,5.980066,0.797342,7.973422,70.099668,2.325581,63,31,45,4,47,490,15,695,9.064748,4.460432,6.474820,0.575540,6.762590,70.503597,2.158273,69,30,45,8,73,565,20,810,8.518519,3.703704,5.555556,0.987654,9.012346,69.753086,2.469136,level_4_qualifications_or_above
3,4,E01000005,City of London 001E,"POLYGON ((533619.062 181402.364, 533639.868 18...",E02000001,City of London 001,E09000001,City of London,E05009308,Portsoken,E12000007,London,Census 2021 (ONS Nomis - Official Census and L...,LSOA 2021,2021-02-21,https://www.nomisweb.co.uk/,18.957771,198,64,120,32,130,387,30,961,20.603538,6.659729,12.486993,3.329865,13.527575,40.270552,3.121748,105,31,50,12,70,188,13,469,22.388060,6.609808,10.660981,2.558635,14.925373,40.085288,2.771855,93,33,70,20,60,199,17,492,18.902439,6.707317,14.227642,4.065041,12.195122,

In [24]:
def determine_dominant_group(row):
    columns = [
 'no_qualifications_female_perc',
 'level_1_and_entry_level_qualifications_female_perc',
 'level_2_qualifications_female_perc',
 'apprenticeship_female_perc',
 'level_3_qualifications_female_perc',
 'level_4_qualifications_or_above_female_perc',
 'other_qualifications_female_perc'
    ]
    
    max_value = max(row[col] for col in columns)
    threshold = max_value - threshold_value
    
    dominant_groups = [col.replace('_perc', '') for col in columns if row[col] >= threshold]
    
    return ', '.join(dominant_groups)

census2021_education_lsoa2021_gdb_df['dominant_education_group_female'] = census2021_education_lsoa2021_gdb_df.apply(determine_dominant_group, axis=1)

# Display the first few rows of the updated dataframe
census2021_education_lsoa2021_gdb_df.head()

,FID,lsoa21cd,lsoa21nm,geometry,msoa21cd,msoa21nm,lad22cd,lad22nm,wd22cd,wd22nm,rgn22cd,rgn22nm,data_source,data_resolution,data_time_period,data_web_link,area_ha,no_qualifications_count,level_1_and_entry_level_qualifications_count,level_2_qualifications_count,apprenticeship_count,level_3_qualifications_count,level_4_qualifications_or_above_count,other_qualifications_count,total_edu_pop_count,no_qualifications_perc,level_1_and_entry_level_qualifications_perc,level_2_qualifications_perc,apprenticeship_perc,level_3_qualifications_perc,level_4_qualifications_or_above_perc,other_qualifications_perc,no_qualifications_female_count,level_1_and_entry_level_qualifications_female_count,level_2_qualifications_female_count,apprenticeship_female_count,level_3_qualifications_female_count,level_4_qualifications_or_above_female_count,other_qualifications_female_count,total_female_edu_pop_count,no_qualifications_female_perc,level_1_and_entry_level_qualifications_female_perc,level_2_qualifications_female_perc,apprenticeship_female_perc,level_3_qualifications_female_perc,level_4_qualifications_or_above_female_perc,other_qualifications_female_perc,no_qualifications_male_count,level_1_and_entry_level_qualifications_male_count,level_2_qualifications_male_count,apprenticeship_male_count,level_3_qualifications_male_count,level_4_qualifications_or_above_male_count,other_qualifications_male_count,total_male_edu_pop_count,no_qualifications_male_perc,level_1_and_entry_level_qualifications_male_perc,level_2_qualifications_male_perc,apprenticeship_male_perc,level_3_qualifications_male_perc,level_4_qualifications_or_above_male_perc,other_qualifications_male_perc,dominant_education_group,dominant_education_group_female
0,1,E01000001,City of London 001A,"POLYGON ((532151.537 181867.433, 532152.500 18...",E02000001,City of London 001,E09000001,City of London,E05009288,Aldersgate,E12000007,London,Census 2021 (ONS Nomis - Official Census and L...,LSOA 2021,2021-02-21,https://www.nomisweb.co.uk/,12.986531,32,27,53,13,86,1127,16,1354,2.363368,1.994092,3.914328,0.960118,6.351551,83.234860,1.181684,15,11,31,5,34,525,7,628,2.388535,1.751592,4.936306,0.796178,5.414013,83.598726,1.114650,17,16,22,8,52,602,9,726,2.341598,2.203857,3.030303,1.101928,7.162534,82.920110,1.239669,level_4_qualifications_or_above,level_4_qualifications_or_above_female
1,2,E01000002,City of London 001B,"POLYGON ((532634.497 181926.016, 532632.048 18...",E02000001,City of London 001,E09000001,City of London,E05009302,Cripplegate,E12000007,London,Census 2021 (ONS Nomis - Official Census and L...,LSOA 2021,2021-02-21,https://www.nomisweb.co.uk/,22.841978,21,28,52,4,87,1086,19,1297,1.619121,2.158828,4.009252,0.308404,6.707787,83.731689,1.464919,15,12,27,1,40,468,6,569,2.636204,2.108963,4.745167,0.175747,7.029877,82.249561,1.054482,6,16,25,3,47,618,13,728,0.824176,2.197802,3.434066,0.412088,6.456044,84.890110,1.785714,level_4_qualifications_or_above,level_4_qualifications_or_above_female
2,3,E01000003,City of London 001C,"POLYGON ((532153.703 182165.155, 532158.250 18...",E02000001,City of London 001,E09000001,City of London,E05009302,Cripplegate,E12000007,London,Census 2021 (ONS Nomis - Official Census and L...,LSOA 2021,2021-02-21,https://www.nomisweb.co.uk/,5.905420,132,61,90,12,120,1055,35,1505,8.770764,4.053156,5.980066,0.797342,7.973422,70.099668,2.325581,63,31,45,4,47,490,15,695,9.064748,4.460432,6.474820,0.575540,6.762590,70.503597,2.158273,69,30,45,8,73,565,20,810,8.518519,3.703704,5.555556,0.987654,9.012346,69.753086,2.469136,level_4_qualifications_or_above,level_4_qualifications_or_above_female
3,4,E01000005,City of London 001E,"POLYGON ((533619.062 181402.364, 533639.868 18...",E02000001,City of London 001,E09000001,City of London,E05009308,Portsoken,E12000007,London,Census 2021 (ONS Nomis - Official Census and L...,LSOA 2021,2021-02-21,https://www.nomisweb.co.uk/,18.957771,198,64,120,32,130,387,30,961,20.603538,6.659729,12.486993,3.329865,13.527575,40.270552,3.121748,105,31,50,12,70,188

In [25]:
def determine_dominant_group(row):
    columns = [
 'no_qualifications_male_perc',
 'level_1_and_entry_level_qualifications_male_perc',
 'level_2_qualifications_male_perc',
 'apprenticeship_male_perc',
 'level_3_qualifications_male_perc',
 'level_4_qualifications_or_above_male_perc',
 'other_qualifications_male_perc'
    ]
    
    max_value = max(row[col] for col in columns)
    threshold = max_value - threshold_value
    
    dominant_groups = [col.replace('_perc', '') for col in columns if row[col] >= threshold]
    
    return ', '.join(dominant_groups)

census2021_education_lsoa2021_gdb_df['dominant_education_group_male'] = census2021_education_lsoa2021_gdb_df.apply(determine_dominant_group, axis=1)

# Display the first few rows of the updated dataframe
census2021_education_lsoa2021_gdb_df.head()

,FID,lsoa21cd,lsoa21nm,geometry,msoa21cd,msoa21nm,lad22cd,lad22nm,wd22cd,wd22nm,rgn22cd,rgn22nm,data_source,data_resolution,data_time_period,data_web_link,area_ha,no_qualifications_count,level_1_and_entry_level_qualifications_count,level_2_qualifications_count,apprenticeship_count,level_3_qualifications_count,level_4_qualifications_or_above_count,other_qualifications_count,total_edu_pop_count,no_qualifications_perc,level_1_and_entry_level_qualifications_perc,level_2_qualifications_perc,apprenticeship_perc,level_3_qualifications_perc,level_4_qualifications_or_above_perc,other_qualifications_perc,no_qualifications_female_count,level_1_and_entry_level_qualifications_female_count,level_2_qualifications_female_count,apprenticeship_female_count,level_3_qualifications_female_count,level_4_qualifications_or_above_female_count,other_qualifications_female_count,total_female_edu_pop_count,no_qualifications_female_perc,level_1_and_entry_level_qualifications_female_perc,level_2_qualifications_female_perc,apprenticeship_female_perc,level_3_qualifications_female_perc,level_4_qualifications_or_above_female_perc,other_qualifications_female_perc,no_qualifications_male_count,level_1_and_entry_level_qualifications_male_count,level_2_qualifications_male_count,apprenticeship_male_count,level_3_qualifications_male_count,level_4_qualifications_or_above_male_count,other_qualifications_male_count,total_male_edu_pop_count,no_qualifications_male_perc,level_1_and_entry_level_qualifications_male_perc,level_2_qualifications_male_perc,apprenticeship_male_perc,level_3_qualifications_male_perc,level_4_qualifications_or_above_male_perc,other_qualifications_male_perc,dominant_education_group,dominant_education_group_female,dominant_education_group_male
0,1,E01000001,City of London 001A,"POLYGON ((532151.537 181867.433, 532152.500 18...",E02000001,City of London 001,E09000001,City of London,E05009288,Aldersgate,E12000007,London,Census 2021 (ONS Nomis - Official Census and L...,LSOA 2021,2021-02-21,https://www.nomisweb.co.uk/,12.986531,32,27,53,13,86,1127,16,1354,2.363368,1.994092,3.914328,0.960118,6.351551,83.234860,1.181684,15,11,31,5,34,525,7,628,2.388535,1.751592,4.936306,0.796178,5.414013,83.598726,1.114650,17,16,22,8,52,602,9,726,2.341598,2.203857,3.030303,1.101928,7.162534,82.920110,1.239669,level_4_qualifications_or_above,level_4_qualifications_or_above_female,level_4_qualifications_or_above_male
1,2,E01000002,City of London 001B,"POLYGON ((532634.497 181926.016, 532632.048 18...",E02000001,City of London 001,E09000001,City of London,E05009302,Cripplegate,E12000007,London,Census 2021 (ONS Nomis - Official Census and L...,LSOA 2021,2021-02-21,https://www.nomisweb.co.uk/,22.841978,21,28,52,4,87,1086,19,1297,1.619121,2.158828,4.009252,0.308404,6.707787,83.731689,1.464919,15,12,27,1,40,468,6,569,2.636204,2.108963,4.745167,0.175747,7.029877,82.249561,1.054482,6,16,25,3,47,618,13,728,0.824176,2.197802,3.434066,0.412088,6.456044,84.890110,1.785714,level_4_qualifications_or_above,level_4_qualifications_or_above_female,level_4_qualifications_or_above_male
2,3,E01000003,City of London 001C,"POLYGON ((532153.703 182165.155, 532158.250 18...",E02000001,City of London 001,E09000001,City of London,E05009302,Cripplegate,E12000007,London,Census 2021 (ONS Nomis - Official Census and L...,LSOA 2021,2021-02-21,https://www.nomisweb.co.uk/,5.905420,132,61,90,12,120,1055,35,1505,8.770764,4.053156,5.980066,0.797342,7.973422,70.099668,2.325581,63,31,45,4,47,490,15,695,9.064748,4.460432,6.474820,0.575540,6.762590,70.503597,2.158273,69,30,45,8,73,565,20,810,8.518519,3.703704,5.555556,0.987654,9.012346,69.753086,2.469136,level_4_qualifications_or_above,level_4_qualifications_or_above_female,level_4_qualifications_or_above_male
3,4,E01000005,City of London 001E,"POLYGON ((533619.062 181402.364, 533639.868 18...",E02000001,City of London 001,E09000001,City of London,E05009308,Portsoken,E12000007,London,Census 2021 (ONS Nomis - Official Census and L...,LSOA 2021,2021-02-21,https://www.

# 9. Upload to geodatabase

In [26]:
# Define database connection parameters
db_host = "PRIORPSRV03"
db_name = "gis"
db_port = "5432"
db_schema = "uk_new"
table_name = "census2021_lsoa2021_education"  # Desired table name
primary_key_column = "lsoa21cd"  # Define the primary key column (change based on your dataset)
geometry_column = "geometry"  # Default geometry column
# Create the database connection string (Windows Authentication - Trusted Connection)
conn_str = f"postgresql+psycopg2://@{db_host}:{db_port}/{db_name}?sslmode=disable"

# Create a SQLAlchemy engine
engine = create_engine(conn_str)

In [27]:
# Ensure the GeoDataFrame has a valid CRS before writing
if census2021_education_lsoa2021_gdb_df.crs is None:
    print("Warning: GeoDataFrame has no CRS. Setting default to EPSG:27700 (British National Grid).")
    census2021_education_lsoa2021_gdb_df.set_crs(epsg=27700, inplace=True)

In [28]:
# Publish the GeoDataFrame to PostGIS
census2021_education_lsoa2021_gdb_df.to_postgis(
    name=table_name,
    con=engine,
    schema=db_schema,
    if_exists="replace",
    index=False,
    dtype={'geometry': 'POLYGON'}  # Ensure geometry is stored as MULTIPOLYGON
)

print(f"Data successfully uploaded to PostGIS: {db_schema}.{table_name}")

# Connect to the database to modify table structure
with engine.connect() as conn:
    # Set Primary Key (if it doesn't exist already)
    alter_pk_query = text(f"""
        ALTER TABLE {db_schema}.{table_name}
        ADD CONSTRAINT {table_name}_pkey PRIMARY KEY ({primary_key_column});
    """)
    
    # Create Spatial Index
    create_spatial_index_query = text(f"""
        CREATE INDEX {table_name}_geom_idx
        ON {db_schema}.{table_name}
        USING GIST ({geometry_column});
    """)

    try:
        conn.execute(alter_pk_query)  # Add Primary Key
        print(f"Primary key set on column: {primary_key_column}")
    except Exception as e:
        print(f"Could not set primary key. It may already exist. Error: {e}")

    try:
        conn.execute(create_spatial_index_query)  # Add Spatial Index
        print(f"Spatial index created for geometry column: {geometry_column}")
    except Exception as e:
        print(f"Could not create spatial index. It may already exist. Error: {e}")

print(f"GeoDataFrame successfully published to PostGIS with Primary Key and Spatial Index: {db_schema}.{table_name}")

Data successfully uploaded to PostGIS: uk_new.census2021_lsoa2021_education
Primary key set on column: lsoa21cd
Spatial index created for geometry column: geometry
GeoDataFrame successfully published to PostGIS with Primary Key and Spatial Index: uk_new.census2021_lsoa2021_education
